# 0-Bosqich: Ma'lumot sifati tahlili va chang bo'roni aniqlash

Bu notebook barcha stansiyalar bo'yicha 10 yillik (2011-2020) meteorologik ma'lumotlarni:
1. Yuklaydi va formatini tekshiradi
2. Bo'sh qiymatlar va sifat muammolarini aniqlaydi
3. Chang bo'roni kunlarini aniqlaydi (tuman filtri bilan)
4. Natijalarni vizualizatsiya qiladi

---

## 1. Kutubxonalar va sozlamalar

In [ ]:
# Kerakli kutubxonalarni o'rnatish (birinchi marta ishga tushirganda)
!pip install pandas xlrd openpyxl matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Kutubxonalar yuklandi!")

In [ ]:
# ============================================================
# SOZLAMALAR — bu yerda yo'lni o'zgartiring!
# ============================================================

# Ma'lumotlar joylashgan papka yo'li
# O'zingizning kompyuteringizga mos yo'lni yozing:
DATA_DIR = "../data/все станции за 10 лет/"

# Agar yuqoridagi yo'l ishlamasa, to'liq yo'lni yozing:
# DATA_DIR = "C:/Users/SizningNom/Documents/dust_storm_all/data/все станции за 10 лет/"
# DATA_DIR = "/home/user/dust_storm_all/data/все станции за 10 лет/"

# Tekshirish
if os.path.exists(DATA_DIR):
    files = [f for f in os.listdir(DATA_DIR) if f.endswith(('.xls', '.XLS', '.xlsx'))]
    print(f"✅ Papka topildi! {len(files)} ta fayl mavjud.")
else:
    print(f"❌ XATO: '{DATA_DIR}' papkasi topilmadi!")
    print("   Yuqoridagi DATA_DIR o'zgaruvchisini to'g'ri yo'lga o'zgartiring.")

## 2. Ma'lumotlarni yuklash

Barcha stansiyalarni bitta katta DataFrame'ga yuklash

In [ ]:
def load_station(filepath):
    """
    Bitta stansiya faylini o'qish.
    Header qatorini avtomatik topadi (Year so'zini qidiradi).
    """
    fname = os.path.basename(filepath)
    
    try:
        # xlsx va xls uchun turli engine
        if fname.endswith('.xlsx'):
            engine = 'openpyxl'
        else:
            engine = 'xlrd'
        
        # Birinchi 5 qatorni o'qib, header qatorini topish
        df_test = pd.read_excel(filepath, header=None, nrows=5, engine=engine)
        
        header_row = None
        for idx in range(len(df_test)):
            row_vals = df_test.iloc[idx].astype(str).str.strip().tolist()
            if 'Year' in row_vals:
                header_row = idx
                break
        
        if header_row is None:
            # Header topilmadi — 0-qatordan boshlash
            header_row = 0
        
        # To'liq o'qish
        df = pd.read_excel(filepath, header=header_row, engine=engine)
        
        # Ustun nomlarini tozalash
        df.columns = [str(c).strip() for c in df.columns]
        
        # Stansiya nomini qo'shish
        station_name = os.path.splitext(fname)[0].strip()
        df['station'] = station_name
        
        return df
    
    except Exception as e:
        print(f"  ⚠️ {fname}: {e}")
        return None

In [ ]:
# Barcha stansiyalarni yuklash
all_data = []
failed_files = []

files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(('.xls', '.XLS', '.xlsx'))])

print(f"Yuklanmoqda: {len(files)} ta fayl...\n")

for i, fname in enumerate(files, 1):
    filepath = os.path.join(DATA_DIR, fname)
    df = load_station(filepath)
    
    if df is not None:
        all_data.append(df)
        print(f"  [{i:2d}/{len(files)}] ✓ {fname} — {len(df)} qator")
    else:
        failed_files.append(fname)

print(f"\n{'='*60}")
print(f"✅ Muvaffaqiyatli: {len(all_data)} stansiya")
if failed_files:
    print(f"❌ Xato: {len(failed_files)} fayl — {failed_files}")

In [ ]:
# Barchasini bitta DataFrame'ga birlashtirish
df_all = pd.concat(all_data, ignore_index=True)

# Sana ustunini yaratish
df_all['Year'] = df_all['Year'].astype(float).astype(int)
df_all['Mon'] = df_all['Mon'].astype(float).astype(int)
df_all['Day'] = df_all['Day'].astype(float).astype(int)
df_all['date'] = pd.to_datetime(df_all[['Year', 'Mon', 'Day']].rename(
    columns={'Year': 'year', 'Mon': 'month', 'Day': 'day'}), errors='coerce')

print(f"\n📊 UMUMIY MA'LUMOT:")
print(f"   Jami qatorlar: {len(df_all):,}")
print(f"   Stansiyalar: {df_all['station'].nunique()}")
print(f"   Davr: {df_all['date'].min().date()} — {df_all['date'].max().date()}")
print(f"   Ustunlar: {df_all.columns.tolist()}")

In [ ]:
# Dastlabki 10 qatorni ko'rish
df_all.head(10)

## 3. Ma'lumot sifati tahlili

In [ ]:
# Asosiy parametrlar uchun bo'sh qiymatlar statistikasi
key_params = ['V', 'Vx8', 'VxG', 'U', 'UN', 'Ed', 'Taav', 'Tg', 'StP', 'SeP']

# Faqat mavjud ustunlarni tanlash
available_params = [p for p in key_params if p in df_all.columns]

# Raqamli qiymatlarga aylantirish
for col in available_params:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Bo'sh qiymatlar foizi
missing_pct = df_all.groupby('station')[available_params].apply(
    lambda x: x.isna().mean() * 100
).round(1)

print("📋 BO'SH QIYMATLAR (%) — har bir stansiya bo'yicha o'rtacha:")
print(missing_pct.mean().round(1).to_string())
print(f"\n📋 Eng muammoli stansiyalar (V ustuni bo'yicha):")
v_missing = missing_pct['V'].sort_values(ascending=False)
print(v_missing[v_missing > 0].to_string())

In [ ]:
# Bo'sh qiymatlar xaritasi (heatmap)
fig, ax = plt.subplots(figsize=(12, 20))

missing_matrix = df_all.groupby('station')[available_params].apply(
    lambda x: x.isna().mean() * 100
).round(1)

sns.heatmap(missing_matrix, annot=False, cmap='YlOrRd', 
            linewidths=0.5, ax=ax, vmin=0, vmax=50)
ax.set_title("Bo'sh qiymatlar (%) — stansiyalar va parametrlar bo'yicha", fontsize=14)
ax.set_xlabel("Parametr")
ax.set_ylabel("Stansiya")
plt.tight_layout()
plt.show()

In [ ]:
# V (ko'rinish) qiymatlarining umumiy statistikasi
print("📊 V (KO'RINISH) STATISTIKASI — stansiya bo'yicha:")
print("="*70)

v_stats = df_all.groupby('station')['V'].describe().round(2)
v_stats = v_stats.sort_values('mean')

print(v_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].to_string())

In [ ]:
# V taqsimoti — bir nechta stansiya uchun
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sample_stations = ['Tashkent', 'NUKUS', 'BUHARA', 'JASLYK', 'MUJNAK', 'KOKARAL']
# Mavjud stansiyalardan tanlash
actual_stations = [s for s in sample_stations if s in df_all['station'].values]

for idx, station in enumerate(actual_stations[:6]):
    ax = axes[idx // 3, idx % 3]
    data = df_all[df_all['station'] == station]['V'].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(x=1.0, color='red', linestyle='--', label='V=1 km')
    ax.axvline(x=2.0, color='orange', linestyle='--', label='V=2 km')
    ax.set_title(f"{station} (o'rt: {data.mean():.1f} km)", fontsize=12)
    ax.set_xlabel('V (km)')
    ax.legend(fontsize=9)

plt.suptitle('Ko\'rinish (V) taqsimoti — tanlangan stansiyalar', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Chang bo'roni kunlarini aniqlash

### Mezonlar (tuman filtrlangan):
- **KUCHLI:** V < 0.5 km + (shamol >= 10 m/s YOKI namlik < 40%)
- **O'RTACHA:** V < 1 km + (shamol >= 8 m/s YOKI namlik < 40%)
- **YENGIL:** V < 2 km + shamol >= 10 m/s + namlik < 40%
- **TUMAN filtri:** V past + shamol < 5 m/s + namlik > 70% → chiqarib tashlanadi

In [ ]:
# Chang bo'roni aniqlash funksiyasi
def detect_dust_storms(df):
    """
    Chang bo'roni kunlarini aniqlash.
    Tuman filtri: past ko'rinish + past shamol + yuqori namlik = TUMAN, CHANG EMAS.
    """
    # Shamol tezligi (VxG ustunlik, aks holda Vx8)
    df = df.copy()
    df['wind'] = df['VxG'].fillna(df['Vx8'])
    df['humidity'] = df['UN'].fillna(df['U'])
    
    # TUMAN FILTRI: past ko'rinish + past shamol + yuqori namlik
    is_fog = (df['V'] < 2) & (df['wind'] < 5) & (df['humidity'] > 70)
    # Shamol va namlik ma'lumoti yo'q + ko'rinish past = noaniq (tuman bo'lishi mumkin)
    is_fog_uncertain = (df['V'] < 2) & (df['wind'].isna()) & (df['humidity'] > 70)
    
    fog_mask = is_fog | is_fog_uncertain
    
    # CHANG BO'RONI mezonlari (tuman emas)
    # KUCHLI: V < 0.5 VA (shamol >= 10 YOKI namlik < 40)
    kuchli = (~fog_mask) & (df['V'] < 0.5) & (
        (df['wind'] >= 10) | (df['humidity'] < 40)
    )
    
    # O'RTACHA: V < 1 VA (shamol >= 8 YOKI namlik < 40) (kuchli bo'lmaganlar)
    ortacha = (~fog_mask) & (~kuchli) & (df['V'] < 1.0) & (
        (df['wind'] >= 8) | (df['humidity'] < 40)
    )
    
    # YENGIL: 1 <= V < 2 VA shamol >= 10 VA namlik < 40
    yengil = (~fog_mask) & (~kuchli) & (~ortacha) & (df['V'] < 2.0) & (
        (df['wind'] >= 10) & (df['humidity'] < 40)
    )
    
    # Daraja ustunini yaratish
    df['dust_severity'] = 'NONE'
    df.loc[yengil, 'dust_severity'] = 'YENGIL'
    df.loc[ortacha, 'dust_severity'] = 'ORTACHA'
    df.loc[kuchli, 'dust_severity'] = 'KUCHLI'
    df.loc[fog_mask, 'dust_severity'] = 'TUMAN'
    
    # Boolean ustun
    df['is_dust'] = df['dust_severity'].isin(['KUCHLI', 'ORTACHA', 'YENGIL'])
    
    return df

In [ ]:
# Chang bo'roni aniqlashni barcha ma'lumotga qo'llash
df_all = detect_dust_storms(df_all)

# Natijalar
dust_counts = df_all['dust_severity'].value_counts()
print("📊 NATIJALAR:")
print("="*50)
print(f"   Jami qatorlar: {len(df_all):,}")
print(f"   \n   TAQSIMOT:")
for severity in ['KUCHLI', 'ORTACHA', 'YENGIL', 'TUMAN', 'NONE']:
    count = dust_counts.get(severity, 0)
    pct = count / len(df_all) * 100
    emoji = {'KUCHLI': '🔴', 'ORTACHA': '🟠', 'YENGIL': '🟡', 'TUMAN': '🌫️', 'NONE': '⚪'}[severity]
    print(f"   {emoji} {severity:<10}: {count:>7,} ({pct:.1f}%)")

print(f"\n   Chang bo'roni (jami): {df_all['is_dust'].sum():,} kun")

## 5. Vizualizatsiya

In [ ]:
# Yillar bo'yicha chang bo'roni soni
# Faqat 2011-2020 yillarni olish (asosiy davr)
df_main = df_all[(df_all['Year'] >= 2011) & (df_all['Year'] <= 2020)].copy()

yearly = df_main[df_main['is_dust']].groupby(['Year', 'dust_severity']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'KUCHLI': '#d32f2f', 'ORTACHA': '#ff9800', 'YENGIL': '#fdd835'}
yearly_plot = yearly[['KUCHLI', 'ORTACHA', 'YENGIL']] if 'YENGIL' in yearly.columns else yearly
yearly_plot.plot(kind='bar', stacked=True, ax=ax, color=[colors.get(c, '#ccc') for c in yearly_plot.columns])

ax.set_title('Chang bo\'roni hodisalari — yillar bo\'yicha (2011-2020)', fontsize=14)
ax.set_xlabel('Yil')
ax.set_ylabel('Hodisalar soni')
ax.legend(title='Daraja')
plt.tight_layout()
plt.show()

In [ ]:
# Oylar bo'yicha (mavsumiylik)
monthly = df_main[df_main['is_dust']].groupby(['Mon', 'dust_severity']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
month_names = ['Yan', 'Fev', 'Mar', 'Apr', 'May', 'Iyn', 
               'Iyl', 'Avg', 'Sen', 'Okt', 'Noy', 'Dek']

monthly_plot = monthly[['KUCHLI', 'ORTACHA', 'YENGIL']] if 'YENGIL' in monthly.columns else monthly
monthly_plot.index = month_names
monthly_plot.plot(kind='bar', stacked=True, ax=ax, 
                  color=[colors.get(c, '#ccc') for c in monthly_plot.columns])

ax.set_title('Chang bo\'roni hodisalari — oylar bo\'yicha (mavsumiylik)', fontsize=14)
ax.set_xlabel('Oy')
ax.set_ylabel('Hodisalar soni (10 yil jami)')
ax.legend(title='Daraja')
plt.tight_layout()
plt.show()

In [ ]:
# Stansiyalar bo'yicha — Top-20
station_dust = df_main[df_main['is_dust']].groupby(['station', 'dust_severity']).size().unstack(fill_value=0)
station_dust['total'] = station_dust.sum(axis=1)
top20 = station_dust.sort_values('total', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 8))
plot_cols = [c for c in ['KUCHLI', 'ORTACHA', 'YENGIL'] if c in top20.columns]
top20[plot_cols].plot(kind='barh', stacked=True, ax=ax,
                      color=[colors.get(c, '#ccc') for c in plot_cols])

ax.set_title('Eng ko\'p chang bo\'roni bo\'lgan stansiyalar (Top-20)', fontsize=14)
ax.set_xlabel('Hodisalar soni (2011-2020)')
ax.set_ylabel('Stansiya')
ax.legend(title='Daraja')
plt.tight_layout()
plt.show()

print("\nTop-20 jadval:")
print(top20[plot_cols + ['total']].to_string())

In [ ]:
# Vaqt bo'yicha trend — Tashkent
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

tashkent = df_main[df_main['station'] == 'Tashkent'].copy()
tashkent = tashkent.set_index('date').sort_index()

# V (ko'rinish)
axes[0].plot(tashkent.index, tashkent['V'], color='steelblue', linewidth=0.5, alpha=0.7)
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='V=1 km')
axes[0].set_ylabel('V (km)')
axes[0].set_title('Tashkent — Ko\'rinish, shamol va namlik (2011-2020)', fontsize=13)
axes[0].legend()

# VxG (shamol)
axes[1].plot(tashkent.index, tashkent['VxG'], color='darkgreen', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='10 m/s')
axes[1].set_ylabel('VxG (m/s)')
axes[1].legend()

# UN (namlik)
axes[2].plot(tashkent.index, tashkent['UN'], color='darkorange', linewidth=0.5, alpha=0.7)
axes[2].axhline(y=40, color='red', linestyle='--', alpha=0.5, label='40%')
axes[2].set_ylabel('UN (%)')
axes[2].set_xlabel('Sana')
axes[2].legend()

# Chang kunlarini belgilash
dust_days = tashkent[tashkent['is_dust']]
for ax in axes:
    for d in dust_days.index:
        ax.axvline(x=d, color='red', alpha=0.05, linewidth=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Chang bo'roni parametrlari o'rtasidagi korrelyatsiya
corr_params = ['V', 'Vx8', 'VxG', 'U', 'UN', 'Ed', 'Taav', 'Tg', 'StP']
corr_params = [p for p in corr_params if p in df_main.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_main[corr_params].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', ax=ax, vmin=-1, vmax=1)
ax.set_title('Parametrlar korrelyatsiyasi', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Natijalarni saqlash

In [ ]:
# Natijalarni CSV formatida saqlash
output_dir = '../analysis/results/'
os.makedirs(output_dir, exist_ok=True)

# 1. Chang bo'roni hodisalari
dust_events = df_main[df_main['is_dust']][[
    'station', 'date', 'dust_severity', 'V', 'VxG', 'Vx8', 'UN', 'U', 'Taav'
]].copy()
dust_events.to_csv(f'{output_dir}dust_storm_events.csv', index=False)
print(f"✓ dust_storm_events.csv — {len(dust_events):,} hodisa")

# 2. Stansiyalar sifati
quality_df = pd.DataFrame({
    'station': df_main['station'].unique(),
})
quality_stats = df_main.groupby('station').agg(
    total_rows=('V', 'count'),
    V_missing_pct=('V', lambda x: x.isna().mean() * 100),
    V_mean=('V', 'mean'),
    V_min=('V', 'min'),
    V_max=('V', 'max'),
    VxG_mean=('VxG', 'mean'),
    dust_total=('is_dust', 'sum'),
).round(2)
quality_stats.to_csv(f'{output_dir}station_quality.csv')
print(f"✓ station_quality.csv — {len(quality_stats)} stansiya")

# 3. To'liq ma'lumot (keyingi bosqichlar uchun)
df_main.to_csv(f'{output_dir}all_stations_daily.csv', index=False)
print(f"✓ all_stations_daily.csv — {len(df_main):,} qator")

print(f"\n💾 Barcha natijalar saqlandi: {output_dir}")

## 7. Xulosa va keyingi qadamlar

### Topilmalar:
1. **Ma'lumot formati:** Kunlik (har bir qator = 1 kun), 2011-2020
2. **V birligi:** km (JASLYK max=23.6, Tashkent max=3.1)
3. **Ma'lumot sifati:** Juda yaxshi — aksariyat stansiyalarda < 5% bo'sh qiymat
4. **Vx8/VxG:** Shamol tezligi m/s da

### Keyingi qadam (1-bosqich):
- Stansiyalar koordinatalarini olish (O'zgidromet yoki qo'lda)
- ERA5 reanaliz ma'lumotlarini ulash
- Sentinel-5P UVAI bilan solishtirish (2018-2020)
- ML model uchun feature engineering

In [ ]:
# Stansiyalar ro'yxati va ularning chang bo'roni statistikasi
summary = df_main.groupby('station').agg(
    kunlar=('date', 'count'),
    V_ortacha=('V', 'mean'),
    shamol_ortacha=('VxG', 'mean'),
    chang_kunlari=('is_dust', 'sum'),
    kuchli=('dust_severity', lambda x: (x == 'KUCHLI').sum()),
    ortacha=('dust_severity', lambda x: (x == 'ORTACHA').sum()),
).round(1)

summary['chang_%'] = (summary['chang_kunlari'] / summary['kunlar'] * 100).round(1)
summary = summary.sort_values('chang_kunlari', ascending=False)

print("📊 BARCHA STANSIYALAR XULOSASI:")
print("="*80)
print(summary.to_string())